# Bài tập về nhà  
Giao diện cho bài toán máy hút bụi với các thuật toán tìm kiếm:  

**Buổi 8:**
- [IDA*](#thuật-toán-iterative-deepening-a-ida)  
    Cách tính các loại chi phí đối với thuật toán này:
    - g(n) = g(parent_node) + cost, với cost tính bằng:
        + Nếu di chuyển vào ô có bụi thì cost = 1
        + Nếu di chuyển vào ô không có bụi thì cost = 3
        Thuật toán sẽ ưu tiên cost bé, tức là ô có bụi.
    - h(n) = khoảng cách manhattan từ robot đến bụi XA NHẤT
    - f(n) = g(n) + h(n)  
    
    Cài đặt:  
    - limit ban đầu tính bằng f(n), alpha = min(f(n))
- [Leo đồi](#thuật-toán-leo-đồi)  
    Cost tính bằng h(n), cài đặt bằng khoảng cách manhattan từ robot đến bụi XA NHẤT
    + [Leo đồi đơn giản](#leo-đồi-đơn-giản)
    + [Leo đồi dốc nhất](#leo-đồi-dốc-nhất)
    + [Leo đồi ngẫu nhiên](#leo-đồi-ngẫu-nhiên)


Ở buổi 5 đã xây dựng các thuật toán:
- BFS cách tiếp cận 1  
- BFS cách tiếp cận 2  
- DFS cách tiếp cận 1  
- DFS cách tiếp cận 2  

Ở buổi 6 đã xây dựng các thuật toán:  
- IDS cách tiếp cận 1  
- IDS cách tiếp cận 2  
- UCS  

Ở buổi 7 đã xây dựng các thuật toán:  
- Greedy Search  
- A*  

**Link github:**  

**BT cá nhân:** Mỗi thuật toán tổ chức thành các module, file gui.ipynb sẽ import các thuật toán và gọi nó khi giao diện chạy thuật toán.  
https://github.com/Duc-Luong060106/vacuum_cleaner_personal_project

In [17]:
# Class định nghĩa các thông tin của trạng thái
class Node:
    def __init__(self, state, parent, action, path_cost):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

In [18]:
# Trả về list là chuỗi đường đi đến node truyền vào
def get_solution(node):
    path = []

    while node != None:
        path.append(node)
        node = node.parent

    path.reverse()
    return path

def print_solution(result):
    if result == None:
        print("Không tìm thấy đường đi!!!")
    else:
        for idx, node in enumerate(result):
            print(f"Bước {idx}: {node.action}")
            for row in node.state:
                print(*row)
            
            print()

        print("Phòng đã được dọn dẹp!!!")

In [19]:
# So sánh trạng thái với trạng thái đích (Không còn bụi)
def compare_state(state, goal_state):
    for x in range(len(state)):
        for y in range(len(state[0])):
            # Vị trí của robot đang đứng đã hút sạch bụi
            if state[x][y] != 'X' and state[x][y] != goal_state[x][y]:
                return False

    return True

In [20]:
def find_vacuum_position(state):
    for x in range(len(state)):
        for y in range(len(state[0])):

            if state[x][y] == 'X':
                return x, y
    
    return None

# Tạo ra các trạng thái con hợp lệ và hành động để sinh ra các trạng thái đó
def gen_actions(state):
    m = len(state)
    n = len(state[0])

    x, y = find_vacuum_position(state)

    actions = []

    if x > 0 and state[x-1][y] != 2:
        up_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x-1}][{y}]" if up_state[x-1][y] == 1 else ""

        up_state[x][y], up_state[x-1][y] = 0, 'X'

        actions.append(("Up" + clean_action, up_state))

    if x < m - 1 and state[x+1][y] != 2:
        down_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x+1}][{y}]" if down_state[x+1][y] == 1 else ""

        down_state[x][y], down_state[x+1][y] = 0, 'X'

        actions.append(("Down" + clean_action, down_state))

    if y > 0 and state[x][y-1] != 2:
        left_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x}][{y-1}]" if left_state[x][y-1] == 1 else ""

        left_state[x][y], left_state[x][y-1] = 0, 'X'

        actions.append(("Left" + clean_action, left_state))

    if y < n - 1 and state[x][y+1] != 2:
        right_state = [row[:] for row in state]
        clean_action = f" và dọn rác ô [{x}][{y+1}]" if right_state[x][y+1] == 1 else ""

        right_state[x][y], right_state[x][y+1] = 0, 'X'

        actions.append(("Right" + clean_action, right_state))

    return actions

In [21]:
# Hàm phụ để so sánh trạng thái đã tồn tại trong frontier 
def is_state_in_frontier(child_state, frontier):
    for node in frontier:
        if node.state == child_state:
            return True
    
    return False

## Thuật toán Iterative Deepening A* (IDA*)

In [22]:
""" Cách tính các loại chi phí đối với thuật toán này:
    - g(n) = g(parent_node) + cost, với cost tính bằng:
        + Nếu di chuyển vào ô có bụi thì cost = 1
        + Nếu di chuyển vào ô không có bụi thì cost = 3
        Thuật toán sẽ ưu tiên cost bé, tức là ô có bụi.
    - h(n) = khoảng cách manhattan từ robot đến bụi XA NHẤT
    - f(n) = g(n) + h(n)
"""
def calculate_cost(state, action):
    x, y = find_vacuum_position(state)

    match(action):
        case action if action.startswith("Up"):
            return 3 if state[x-1][y] == 0 else 1
        case action if action.startswith("Down"):
            return 3 if state[x+1][y] == 0 else 1
        case action if action.startswith("Left"):
            return 3 if state[x][y-1] == 0 else 1
        case action if action.startswith("Right"):
            return 3 if state[x][y+1] == 0 else 1


def calculate_heuristic_cost(state):
    x, y = find_vacuum_position(state)
    heuristic_cost = 0

    for i in range(len(state)):
        for j in range(len(state[0])):
            if state[i][j] == 1:
                manhattan_distance = abs(x-i) + abs(y-j)
                heuristic_cost = max(heuristic_cost, manhattan_distance) if heuristic_cost != 0 else manhattan_distance 
    
    return heuristic_cost

In [23]:
def pop_lowest_cost_node(frontier):
    lowest_cost_node = min(frontier, key=lambda node: node.path_cost['f'])
    frontier.remove(lowest_cost_node)

    return lowest_cost_node

In [24]:
def iterative_deepening_a_star(initial, goal):
    limit = calculate_heuristic_cost(initial)   # g(n) = 0 --> f(n) = h(n)

    while True:
        result = cost_limited_search(initial, goal, limit)

        if not isinstance(result, tuple):
            if result == "failure":
                return None
            else: 
                return result
        
        limit += result[1]


def cost_limited_search(initial, goal, l):
    start = Node(initial, None, None, None)
    h_start = calculate_heuristic_cost(initial)
    start.path_cost = {'g': 0, 'h': h_start, 'f':h_start}

    if compare_state(start.state, goal):
        return get_solution(start)
    
    frontier = []
    frontier.append(start)

    reached = {}

    result = "failure"
    alpha = None

    while frontier:
        node = pop_lowest_cost_node(frontier)
        
        if compare_state(node.state, goal):
            return get_solution(node)
        
        if node.path_cost['f'] > l:
            alpha = min(node.path_cost['f'], alpha) if alpha != None else node.path_cost['f']
            result = ("cutoff", alpha)
            continue
        
        state_tuple = tuple(tuple(row) for row in node.state)
        reached[state_tuple] = node.path_cost['g']

        for action, child_state in gen_actions(node.state):

            g_new = node.path_cost['g'] + calculate_cost(node.state, action)
            h_n = calculate_heuristic_cost(child_state)
            if g_new + h_n > l:
                alpha = min(g_new + h_n, alpha) if alpha != None else (g_new + h_n)
                result = ("cutoff", alpha)
                continue

            child_state_tup = tuple(tuple(row) for row in child_state)

            if child_state_tup in reached:
                if g_new >= reached[child_state_tup]:
                    continue
                else:
                    del reached[child_state_tup]

            if is_state_in_frontier(child_state, frontier):
                child_in_frontier = list(n for n in frontier if n.state == child_state)[0]
                
                if g_new < child_in_frontier.path_cost['g']:
                    child_in_frontier.parent = node
                    child_in_frontier.path_cost['g'] = g_new
                    child_in_frontier.path_cost['f'] = child_in_frontier.path_cost['g'] + child_in_frontier.path_cost['h']

            elif not is_state_in_frontier(child_state, frontier) and child_state_tup not in reached:
                child = Node(child_state, node, action, None)
                g_n = g_new
                h_n = calculate_heuristic_cost(child_state)
                child.path_cost = {'g': g_n, 'h': h_n, 'f':g_n + h_n}

                frontier.append(child)
        
    return result

In [25]:
start =  [['X', 0 , 1 , 1],
          [ 0 , 2 , 0 , 0],
          [ 1 , 0 , 1 , 2],
          [ 2 , 1 , 1 , 1],
          [ 1 , 0 , 1 , 1]]

goal = [[0, 0, 0, 0],
        [0, 2, 0, 0],
        [0, 0, 0, 2],
        [2, 0, 0, 0],
        [0, 0, 0, 0]]

result = iterative_deepening_a_star(start, goal)
print_solution(result)


Bước 0: None
X 0 1 1
0 2 0 0
1 0 1 2
2 1 1 1
1 0 1 1

Bước 1: Down
0 0 1 1
X 2 0 0
1 0 1 2
2 1 1 1
1 0 1 1

Bước 2: Down và dọn rác ô [2][0]
0 0 1 1
0 2 0 0
X 0 1 2
2 1 1 1
1 0 1 1

Bước 3: Right
0 0 1 1
0 2 0 0
0 X 1 2
2 1 1 1
1 0 1 1

Bước 4: Down và dọn rác ô [3][1]
0 0 1 1
0 2 0 0
0 0 1 2
2 X 1 1
1 0 1 1

Bước 5: Down
0 0 1 1
0 2 0 0
0 0 1 2
2 0 1 1
1 X 1 1

Bước 6: Left và dọn rác ô [4][0]
0 0 1 1
0 2 0 0
0 0 1 2
2 0 1 1
X 0 1 1

Bước 7: Right
0 0 1 1
0 2 0 0
0 0 1 2
2 0 1 1
0 X 1 1

Bước 8: Right và dọn rác ô [4][2]
0 0 1 1
0 2 0 0
0 0 1 2
2 0 1 1
0 0 X 1

Bước 9: Right và dọn rác ô [4][3]
0 0 1 1
0 2 0 0
0 0 1 2
2 0 1 1
0 0 0 X

Bước 10: Up và dọn rác ô [3][3]
0 0 1 1
0 2 0 0
0 0 1 2
2 0 1 X
0 0 0 0

Bước 11: Left và dọn rác ô [3][2]
0 0 1 1
0 2 0 0
0 0 1 2
2 0 X 0
0 0 0 0

Bước 12: Up và dọn rác ô [2][2]
0 0 1 1
0 2 0 0
0 0 X 2
2 0 0 0
0 0 0 0

Bước 13: Up
0 0 1 1
0 2 X 0
0 0 0 2
2 0 0 0
0 0 0 0

Bước 14: Up và dọn rác ô [0][2]
0 0 X 1
0 2 0 0
0 0 0 2
2 0 0 0
0 0 0 0

Bước 15: 

## Thuật toán leo đồi

In [26]:
# Nhóm thuật toán này sử dụng chi phí heuristic (h(n)), được cài đặt là khoảng cách heuristic XA NHẤT từ robot đến rác.
def calculate_heuristic_cost(state):
    x, y = find_vacuum_position(state)
    heuristic_cost = 0

    for i in range(len(state)):
        for j in range(len(state[0])):
            if state[i][j] == 1:
                manhattan_distance = abs(x-i) + abs(y-j)
                heuristic_cost = max(heuristic_cost, manhattan_distance) if heuristic_cost != 0 else manhattan_distance 
    
    return heuristic_cost

### Leo đồi đơn giản

In [27]:
def simple_hill_climbing(initial, goal):
    current_node = Node(initial, None, None, calculate_heuristic_cost(initial))
    
    while True:
        if compare_state(current_node.state, goal):
            return get_solution(current_node)

        found_better_neighbor = False

        for action, child_state in gen_actions(current_node.state):
            child_cost = calculate_heuristic_cost(child_state)
            if child_cost < current_node.path_cost:
                current_node = Node(child_state, current_node, action, child_cost)
                found_better_neighbor = True
                break
            
        if not found_better_neighbor:
            return None

In [28]:
start =  [['X', 0],
          [0, 1]]

goal = [[0, 0],
        [0, 0]] 

result = simple_hill_climbing(start, goal)

print_solution(result)

Bước 0: None
X 0
0 1

Bước 1: Down
0 0
X 1

Bước 2: Right và dọn rác ô [1][1]
0 0
0 X

Phòng đã được dọn dẹp!!!


### Leo đồi dốc nhất

In [29]:
import random

def steepest_ascent_hill_climbing(initial, goal):
    current_node = Node(initial, None, None, calculate_heuristic_cost(initial))
    
    while True:
        if compare_state(current_node.state, goal):
            return get_solution(current_node)

        best_neighbor = {"cost": current_node.path_cost-1, "child_info": []}

        for action, child_state in gen_actions(current_node.state):
            child_cost = calculate_heuristic_cost(child_state)

            if child_cost < best_neighbor["cost"]:
                best_neighbor["child_info"] = [(action, child_state)]
                best_neighbor["cost"] = child_cost

            elif child_cost == best_neighbor["cost"]:
                best_neighbor["child_info"].append((action, child_state))
                
        if best_neighbor["child_info"]:
            action, child_state = random.choice(best_neighbor["child_info"])
            current_node = Node(child_state, current_node, action, best_neighbor["cost"])
            
        else:
            return None

In [30]:
start =  [['X', 0],
          [0, 1]]

goal = [[0, 0],
        [0, 0]] 

result = steepest_ascent_hill_climbing(start, goal)

print_solution(result)

Bước 0: None
X 0
0 1

Bước 1: Right
0 X
0 1

Bước 2: Down và dọn rác ô [1][1]
0 0
0 X

Phòng đã được dọn dẹp!!!


### Leo đồi ngẫu nhiên

In [31]:
import random

def stochastic_hill_climbing(initial, goal):
    current_node = Node(initial, None, None, calculate_heuristic_cost(initial))
    
    while True:
        if compare_state(current_node.state, goal):
            return get_solution(current_node)

        better_neighbor = []

        for action, child_state in gen_actions(current_node.state):
            child_cost = calculate_heuristic_cost(child_state)

            if child_cost < current_node.path_cost:
                better_neighbor.append((action, child_state, child_cost))
                
        if better_neighbor:
            action, child_state, child_cost = random.choice(better_neighbor)
            current_node = Node(child_state, current_node, action, child_cost)
            
        else:
            return None

In [32]:
start =  [['X', 0],
          [0, 1]]

goal = [[0, 0],
        [0, 0]] 

result = stochastic_hill_climbing(start, goal)

print_solution(result)

Bước 0: None
X 0
0 1

Bước 1: Down
0 0
X 1

Bước 2: Right và dọn rác ô [1][1]
0 0
0 X

Phòng đã được dọn dẹp!!!
